# Embedding sweep — model × prompt × structure LMC over the full corpus

Computes the LMC grid that feeds the expanded Stan battery:

| | |
|---|---|
| **Models** | MuQ-MuLan · LAION-CLAP · Microsoft CLAP · CLaMP 3 (4) |
| **Prompts** | raw · “a song[ segment] that contains the lyrics …” · “… representing the idea of …” (3) |
| **Methods** | song · seg_chorus · seg_nonchorus · line ±1 · line ±5 · line ±10 (6 scalars) |

→ **12 embeddings × 5 Stan structures = 60 fits** (song + 3 line windows as track models;
chorus/verse as one segment model). No curvature.

`LMC = cosine(audio, prompt(text))` per (song, model, prompt, method); line methods are the
**mean** cosine over a song's lines. It reuses your existing corpus, audio, MERT controls,
chorus flags, and **native LRCLIB line timestamps** (no Whisper — every corpus song already
has synced lyrics). **Fully resumable**: a (song, model) already done is skipped, so you can
stop/restart and run one model at a time.

> Assumes the observational pipeline has already produced the corpus + audio +
> `results/master_results.csv` (with `mert_pc*` controls). This notebook only adds the new
> model×prompt embeddings on top of that.

## Prerequisites
Run in the **`lmc` conda env**. `pip install msclap` (Microsoft CLAP). For CLaMP 3, set its
env vars in the first cell below (same as the validation notebook).

In [1]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ── CLaMP 3 config — MUST be set BEFORE importing sweep_lib (read at import). ──
# Comment both out to skip CLaMP 3 (its columns will just be absent → those fits drop).
os.environ['LMCVAL_CLAMP3_REPO']   = os.path.expanduser('~/Desktop/clamp3')
os.environ['LMCVAL_CLAMP3_PYTHON'] = '/opt/anaconda3/envs/clamp3/bin/python'   # your clamp3 env python

here = Path.cwd()
root = here if (here / 'src').exists() else here.parent      # repo root or sweep/
for p in (root / 'src', root / 'validation', root / 'sweep'):
    if str(p) not in sys.path: sys.path.insert(0, str(p))

from sweep_lib import config, compute, build_master
from lmc.utils import setup_logging
setup_logging()   # IMPORTANT: turns on the per-song progress logs (else it looks hung)
print(config.summary())

Models   : mulan, clap, msclap, clamp3
Prompts  : raw, contains, idea
Methods  : song, seg_chorus, seg_nonchorus, line_buf1, line_buf5, line_buf10
Grid     : 4×3 embeddings × 5 structures = 60 Stan fits
Project DB : /Users/budge.13/Desktop/Music Analysis/data/project.db
Sweep CSV  : /Users/budge.13/Desktop/Music Analysis/results/master_results_sweep.csv
Stan output: /Users/budge.13/Desktop/Music Analysis/sweep/output


## 0. Prerequisite data (compute only if missing — all resumable)
The sweep needs: audio (assumed present), the MERT control vectors, and a `master_results.csv`
that carries the `mert_pc*` controls + outcome/genre. These come from the observational
pipeline. The cells below are safe to run — each skips work already done. Skip them if your
corpus is already fully built.

In [2]:
# Controls / outcome come from the existing pipeline. Uncomment to (re)build if needed.
from lmc import config as lmc_config, db as projdb, mert as mert_mod, combine
with projdb.connect() as c:
    n_audio = projdb.count(c, 'audio', "status='done'")
    n_mert  = projdb.count(c, 'mert')
print(f'songs with audio: {n_audio}   |   MERT vectors cached: {n_mert}')

# mert_mod.extract_pending()        # <- compute MERT control vectors if n_mert < n_audio (heavy)
# combine.build_master()            # <- (re)write results/master_results.csv incl. mert_pc* controls
assert (lmc_config.RESULTS_DIR / 'master_results.csv').exists(), \
    'Run combine.build_master() first — the sweep reuses its controls/outcome.'

songs with audio: 1075   |   MERT vectors cached: 1075


## 1. Compute the grid — one model at a time (recommended)
**Reuse of your existing embeddings.** MuLan and CLAP are already cached in
`data/embeddings/` from the observational pipeline, so the sweep **reuses their audio
embeddings** (audio never depends on the prompt) and the **raw-prompt text** — it only
re-embeds the short *contains/idea* wrapped text. So `raw` costs zero model calls and
those two models are fast. The per-song log shows `cache` vs `fresh`. (Sanity check: the
sweep's `mulan_raw_*` / `clap_raw_*` columns should equal your existing `mulan_*` / `clap_*`
columns exactly — same cached vectors.) **MS-CLAP and CLaMP 3 are new models with no cache**,
so they compute from scratch. MERT isn't an LMC model here — it's your controls (reused via
`master_results.csv`); CLaMP 3 uses its own separate MERT-95M internally.

Each fast model logs **one line per song**; CLaMP 3 logs per chunk. Heartbeat example:
`[mulan] 3/10  track 41827  cache → 18 rows  (0.6s/song avg)`.

**Smoke test first:** `limit=10` to confirm the path end-to-end. It's resumable, so
stopping/restarting is free.

In [3]:
# Smoke test on 10 songs (writes real rows; resumable so it just becomes a head start):
compute.compute_model('mulan', limit=10)
compute.coverage()

11:07:46 | INFO | sweep_lib.compute | [mulan] 10 songs pending — loading model…
11:07:50 | INFO | lmc.embeddings | Loading MuQ-MuLan on mps…
11:07:55 | INFO | lmcval.models |   loaded MuQ-MuLan.
11:07:55 | INFO | sweep_lib.compute | [mulan] model loaded; starting.
11:07:58 | INFO | sweep_lib.compute |   [mulan] 1/10  track 222237     cache → 18 rows  (2.6s/song avg)
11:07:59 | INFO | sweep_lib.compute |   [mulan] 2/10  track 252527     cache → 18 rows  (2.0s/song avg)
11:08:02 | INFO | sweep_lib.compute |   [mulan] 3/10  track 281140     cache → 15 rows  (2.3s/song avg)
11:08:03 | INFO | sweep_lib.compute |   [mulan] 4/10  track 326218     cache → 18 rows  (2.0s/song avg)
11:08:04 | INFO | sweep_lib.compute |   [mulan] 5/10  track 434739     cache → 18 rows  (1.8s/song avg)
11:08:05 | INFO | sweep_lib.compute |   [mulan] 6/10  track 489995     cache → 18 rows  (1.6s/song avg)
11:08:06 | INFO | sweep_lib.compute |   [mulan] 7/10  track 531194     cache → 18 rows  (1.5s/song avg)
11:08:0

{'songs_with_audio': 1075,
 'done': {'mulan': 14, 'clap': 0, 'msclap': 0, 'clamp3': 0}}

In [ ]:
# Full runs — fast models:
compute.compute_model('mulan')
compute.compute_model('clap')
compute.compute_model('msclap')
compute.coverage()

11:08:15 | INFO | sweep_lib.compute | [mulan] 1061 songs pending — loading model…
11:08:15 | INFO | lmc.embeddings | Loading MuQ-MuLan on mps…
11:08:22 | INFO | lmcval.models |   loaded MuQ-MuLan.
11:08:22 | INFO | sweep_lib.compute | [mulan] model loaded; starting.
11:08:24 | INFO | sweep_lib.compute |   [mulan] 1/1061  track 697163     cache → 18 rows  (1.7s/song avg)
11:08:26 | INFO | sweep_lib.compute |   [mulan] 2/1061  track 698853     cache → 18 rows  (1.9s/song avg)
11:08:28 | INFO | sweep_lib.compute |   [mulan] 3/1061  track 834390     cache → 18 rows  (1.9s/song avg)
11:08:28 | INFO | sweep_lib.compute |   [mulan] 4/1061  track 895865     cache → 18 rows  (1.6s/song avg)
11:08:31 | INFO | sweep_lib.compute |   [mulan] 5/1061  track 1003269    cache → 18 rows  (1.8s/song avg)
11:08:32 | INFO | sweep_lib.compute |   [mulan] 6/1061  track 1050842    cache → 18 rows  (1.7s/song avg)
11:08:34 | INFO | sweep_lib.compute |   [mulan] 7/1061  track 1074484    cache → 18 rows  (1.7s/s

In [ ]:
# CLaMP 3 — slow; resumable; logs per chunk. chunk_size trades model-reloads vs memory
# (bigger = fewer subprocess reloads but more RAM held at once; 8 is a safe default here).
compute.compute_model('clamp3', chunk_size=8)
compute.coverage()

## 2. Assemble `master_results_sweep.csv`
Merges the 72 sweep LMC columns onto the existing master's outcome / genre / MERT controls.

In [ ]:
summary = build_master.build()
print(summary)
print(f"\nGlobal complete-case corpus (what the 60 Stan fits use): "
      f"{summary['global_complete_case']} songs")

In [ ]:
# Per (model, prompt, method) coverage — spot any gaps (e.g. CLaMP 3 song failures):
build_master.missing_report()

## 3. Next: fit the models + report (in RStudio)

```bash
Rscript sweep/R/run_models_sweep.R            # 60 fits: 4 chains × (1000 warmup + 1000 sampling), MERT controls
```
Then render `sweep/lmc_report_sweep.qmd` for the model-comparison + posterior-check report.

**Notes**
- Resumable: re-running any `compute_model` skips songs already done for that model.
- The Stan battery fits on **one global complete-case corpus** (a song must have all 60
  measures) so every LOO is comparable — watch the `global_complete_case` count above; heavy
  CLaMP 3 / MS-CLAP loss shrinks it for everyone.
- Offline plumbing test: `cd sweep && python -m sweep_lib.selftest`.